In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from collections import deque
from typing import List, Dict, Optional
import statistics

# Input Models

@dataclass(frozen=True)
class IngestedTelemetryRecord:
    timestamp: datetime
    rack_temps: List[float]
    airflow_proxy: float
    chiller_power: float
    facility_power: float
    ups_load: float
    tariff: float
    raw_format: str

@dataclass
class ValidatedTelemetryRecord:
    data: IngestedTelemetryRecord
    is_valid: bool
    validation_flags: List[str]
    reason_codes: List[str]

# Output Model

@dataclass
class SystemStateRecord:
    """System state and metrics."""
    current_data: ValidatedTelemetryRecord
    last_safe_data: Optional[IngestedTelemetryRecord]
    
    # Derived Metrics
    delta_t_trend: float                # °C/min
    avg_rack_temp_15m: float            # 15m avg
    avg_facility_power_1h: float        # 1h avg
    is_stable: bool                     # Trend limits
    window_15m_count: int               # Buffer count

class EnerviaStateStore:
    """Time-Series State Manager."""

    def __init__(self):
        # 1-min sampling buffers
        self.buffer_15m = deque(maxlen=15)
        self.buffer_1h = deque(maxlen=60)
        self.buffer_24h = deque(maxlen=1440)
        
        self.last_known_safe_state: Optional[IngestedTelemetryRecord] = None

    def update_state(self, record: ValidatedTelemetryRecord) -> SystemStateRecord:
        """Update state and metrics."""
        # Store safe state 
        if record.is_valid:
            self.last_known_safe_state = record.data
            # Update buffers
            self.buffer_15m.append(record.data)
            self.buffer_1h.append(record.data)
            self.buffer_24h.append(record.data)

        return self.get_current_state(record)

    def _calculate_delta_t_trend(self) -> float:
        """Compute Delta T trend."""
        if len(self.buffer_15m) < 2:
            return 0.0
        
        latest = statistics.mean(self.buffer_15m[-1].rack_temps)
        earliest = statistics.mean(self.buffer_15m[0].rack_temps)
        
        # Calculate slope
        time_diff_min = (self.buffer_15m[-1].timestamp - self.buffer_15m[0].timestamp).total_seconds() / 60
        
        return (latest - earliest) / time_diff_min if time_diff_min > 0 else 0.0

    def get_current_state(self, current_record: ValidatedTelemetryRecord) -> SystemStateRecord:
        """Generate state record."""
        
        # Moving averages
        avg_temp_15m = 0.0
        if self.buffer_15m:
            valid_temps = [
                statistics.mean(r.rack_temps) 
                for r in self.buffer_15m if r.rack_temps
            ]
            if valid_temps:
                avg_temp_15m = statistics.mean(valid_temps)

        avg_power_1h = 0.0
        if self.buffer_1h:
            avg_power_1h = statistics.mean([r.facility_power for r in self.buffer_1h])

        delta_t = self._calculate_delta_t_trend()

        # Stability check
        is_stable = delta_t < 0.5  # < 0.5°C/min

        return SystemStateRecord(
            current_data=current_record,
            last_safe_data=self.last_known_safe_state,
            delta_t_trend=delta_t,
            avg_rack_temp_15m=avg_temp_15m,
            avg_facility_power_1h=avg_power_1h,
            is_stable=is_stable,
            window_15m_count=len(self.buffer_15m)
        )